In [9]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

# %pip install -U langchain langchain-core langchain-google-genai langchain-community langgraph tavily-python python-dotenv

  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.11
    Uninstalling langgraph-1.2.11:
      Successfully uninstalled langgraph-1.2.11
Note: you may need to restart the kernel to use updated packages.


C:\Users\khavy\AppData\Local\Temp\ipykernel_3740\2441863494.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Retrieval Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by giving them access to external, up-to-date, and domain-specific information.

In essence, RAG addresses some key limitations of traditional LLMs, such as:
1.  **Knowledge Cutoff:** LLMs are trained on data up to a certain point and don't inherently know about recent events or new information.
2.  **Hallucination:** LLMs can sometimes generate plausible but factually incorrect information.
3.  **Lack of Specificity:** They might provide general answers when a highly specific, authoritative answer is required.
4.  **No Source Attribution:** They can't tell you where they got their information from.

### How RAG Works (The Process):

RAG combines two main components: a **retrieval system** and a **generation system** (the LLM). Here's a simplified breakdown of the process:

1.  **User Query:** A user asks a question or provides a prompt.
2.  **Retrieval Phase:**
    *   Th

C:\Users\khavy\AppData\Local\Temp\ipykernel_3740\2441863494.py:78: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(


[{'title': 'Recent Developments in Generative AI', 'url': 'https://www.youtube.com/watch?v=uF2a5kThM3g', 'content': "# Recent Developments in Generative AI\n## Channel: Accenture (verified)\n93.6K subscribers\n65 likes\n\n### Description\n3,951 views\nPosted: 2025-03-28\nIt seems like every day, there is a new advance in generative AI. Large multimodal  models (LMMs) and Small Language Models (SMLs) are becoming popular, while Retrieval-Augmented Generation (RAG) is boosting the accuracy of Gen AI models. Then, there are open-source models democratizing access to Gen AI, though not all of them are available or customizable for free.\nWith all these and more, learn about the exciting evolution of Gen AI and why the future is looking very interesting indeed! [...] ### Transcript\n[0:00] [Music]\n[0:06] the era of generative AI started on November 30th 2022 with the launch of chat GPT and ever since the generative\n[0:15] AI landscape has been expanding and evolving at a record Pace much 

In [10]:
# ============================================================
# 1. IMPORT LIBRARIES AND LOAD API KEYS
# ============================================================

import os
from dotenv import load_dotenv

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain tools
from langchain_core.tools import Tool, StructuredTool

# Tavily web search
from langchain_community.tools.tavily_search import TavilySearchResults

# Modern ReAct agent
from langgraph.prebuilt import create_react_agent

# Pydantic for structured tool input
from pydantic import BaseModel


# ------------------------------------------------------------
# Load variables from .env
# ------------------------------------------------------------

load_dotenv(".env")


# ------------------------------------------------------------
# Read API keys
# ------------------------------------------------------------

google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")


# ------------------------------------------------------------
# Check whether API keys exist
# ------------------------------------------------------------

if not google_api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found. "
        "Add it to your .env file."
    )

if not tavily_api_key:
    raise ValueError(
        "TAVILY_API_KEY not found. "
        "Add it to your .env file."
    )


print("✅ API keys loaded successfully.")

✅ API keys loaded successfully.


In [11]:
# ============================================================
# 2. INITIALIZE GEMINI
# ============================================================
# We are replacing the old:
#
# ChatOpenAI(model="gpt-3.5-turbo")
#
# with Google's Gemini chat model.
# ============================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=google_api_key,
    temperature=0.2
)

print("✅ Gemini model initialized.")

✅ Gemini model initialized.


In [12]:
# ============================================================
# 3. SIMPLE QA TOOL
# ============================================================

from langchain_core.prompts import PromptTemplate


# ------------------------------------------------------------
# Create a prompt
# ------------------------------------------------------------

qa_prompt = PromptTemplate.from_template(
    """
    Answer the following question clearly and accurately.

    Question:
    {question}

    Answer:
    """
)


# ------------------------------------------------------------
# Modern LangChain chain
# ------------------------------------------------------------
# Old:
# LLMChain(llm=llm, prompt=qa_prompt)
#
# New:
# prompt | llm
# ------------------------------------------------------------

qa_chain = qa_prompt | llm


# ------------------------------------------------------------
# Function used by the tool
# ------------------------------------------------------------

def qa_tool_fn(question: str) -> str:
    
    response = qa_chain.invoke({
        "question": question
    })
    
    return response.content


# ------------------------------------------------------------
# Create the tool
# ------------------------------------------------------------

qa_tool = Tool(
    name="Simple_QA",
    func=qa_tool_fn,
    description="""
    Answer general questions clearly.
    Use this tool when a question can be answered
    using the language model without web search.
    """
)


print("✅ QA tool created.")

✅ QA tool created.


In [13]:
# ============================================================
# 4. WEB SEARCH TOOL
# ============================================================
# Tavily allows the agent to search the internet for
# current information.
# ============================================================

tavily_search = TavilySearchResults(
    max_results=3
)


# ------------------------------------------------------------
# Wrap Tavily as a LangChain Tool
# ------------------------------------------------------------

search_tool = Tool(
    name="Web_Search",
    func=tavily_search.invoke,
    description="""
    Search the internet for current information,
    recent updates, facts, documentation, or information
    that may not be available in the model's knowledge.
    """
)


# ------------------------------------------------------------
# Combine tools
# ------------------------------------------------------------

tools = [
    qa_tool,
    search_tool
]


print("✅ Tools created:")
for tool in tools:
    print("-", tool.name)

✅ Tools created:
- Simple_QA
- Web_Search


In [14]:
# ============================================================
# 5. ZERO-SHOT REACT AGENT
# ============================================================
#
# OLD:
#
# agent = initialize_agent(
#     tools=tools,
#     llm=llm,
#     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=True
# )
#
# NEW:
#
# LangGraph provides the modern ReAct implementation.
# ============================================================

agent = create_react_agent(
    model=llm,
    tools=tools
)


# ------------------------------------------------------------
# Run the agent
# ------------------------------------------------------------
# Old:
# agent.run("...")
#
# New:
# agent.invoke({
#     "messages": [...]
# })
# ------------------------------------------------------------

result = agent.invoke({
    "messages": [
        (
            "user",
            "Summarize LangChain in 2 lines, then tell me who created it."
        )
    ]
})


# ------------------------------------------------------------
# Display the final response
# ------------------------------------------------------------

print("✅ Result:")
print(result["messages"][-1].content)

C:\Users\khavy\AppData\Local\Temp\ipykernel_3740\3997343502.py:19: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


✅ Result:



In [15]:
# ============================================================
# 6. CONVERSATIONAL REACT
# ============================================================
#
# Instead of the old:
#
# ConversationBufferMemory
#
# we maintain the conversation through messages.
# ============================================================


# ------------------------------------------------------------
# Start conversation history
# ------------------------------------------------------------

chat_history = [
    (
        "user",
        "What is LangChain?"
    )
]


# ------------------------------------------------------------
# First question
# ------------------------------------------------------------

result = agent.invoke({
    "messages": chat_history
})


assistant_response = result["messages"][-1].content

print("Assistant:")
print(assistant_response)


# ------------------------------------------------------------
# Add assistant response to conversation
# ------------------------------------------------------------

chat_history.append(
    (
        "assistant",
        assistant_response
    )
)


# ------------------------------------------------------------
# Ask a follow-up question
# ------------------------------------------------------------

chat_history.append(
    (
        "user",
        "Who created it?"
    )
)


result = agent.invoke({
    "messages": chat_history
})


print("\nAssistant:")
print(result["messages"][-1].content)

Assistant:
LangChain is an open-source framework designed to simplify the development of applications powered by Large Language Models (LLMs). It acts as a set of tools, libraries, and abstractions that allow developers to connect LLMs with external data sources, computation, and other agents, enabling them to build more complex and powerful applications.

Its core purposes include:
*   **Orchestration:** Chaining together different components like LLM calls, API calls, and database queries.
*   **Data Augmentation:** Enabling LLMs to interact with external data for more accurate and context-rich responses.
*   **Agentic Behavior:** Allowing LLMs to act as "agents" that can decide which tools to use and in what order to accomplish a user's goal.

Key components of LangChain include:
*   **Models:** Abstractions for interacting with various LLM providers.
*   **Prompts:** Tools for managing and templating prompts to LLMs.
*   **Chains:** Sequences of calls combining LLMs with other comp

In [16]:
# ============================================================
# 6. CONVERSATIONAL REACT
# ============================================================
#
# Instead of the old:
#
# ConversationBufferMemory
#
# we maintain the conversation through messages.
# ============================================================


# ------------------------------------------------------------
# Start conversation history
# ------------------------------------------------------------

chat_history = [
    (
        "user",
        "What is LangChain?"
    )
]


# ------------------------------------------------------------
# First question
# ------------------------------------------------------------

result = agent.invoke({
    "messages": chat_history
})


assistant_response = result["messages"][-1].content

print("Assistant:")
print(assistant_response)


# ------------------------------------------------------------
# Add assistant response to conversation
# ------------------------------------------------------------

chat_history.append(
    (
        "assistant",
        assistant_response
    )
)


# ------------------------------------------------------------
# Ask a follow-up question
# ------------------------------------------------------------

chat_history.append(
    (
        "user",
        "Who created it?"
    )
)


result = agent.invoke({
    "messages": chat_history
})


print("\nAssistant:")
print(result["messages"][-1].content)

Assistant:
LangChain is an open-source development framework designed to simplify the creation of applications powered by large language models (LLMs). It acts as a bridge, allowing developers to connect LLMs with external data sources, computational tools, and other components. This enables the building of more complex, context-aware, and interactive LLM applications than would be possible with LLMs alone.

Essentially, LangChain helps overcome the limitations of standalone LLMs (like their lack of real-time information, inability to perform actions, or limited context window) by providing a structured way to:

1.  **Integrate LLMs with External Data:** Connect LLMs to your own data (documents, databases, APIs) to provide them with up-to-date or domain-specific information, often through a process called Retrieval Augmented Generation (RAG).
2.  **Enable LLMs to Interact with Tools:** Allow LLMs to use external tools (like search engines, calculators, code interpreters, or custom APIs

In [17]:
# ============================================================
# 7. CHAT CONVERSATIONAL REACT
# ============================================================
#
# Gemini is already a chat model.
#
# Therefore we don't need:
#
# AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION
#
# We simply pass a conversation to the ReAct agent.
# ============================================================


chat_history = [
    (
        "user",
        "Plan a 3-step study path for LangChain."
    )
]


result = agent.invoke({
    "messages": chat_history
})


print("✅ Result:")
print(result["messages"][-1].content)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19h48m11.268796627s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

In [18]:
# ============================================================
# 8. STRUCTURED TOOL
# ============================================================
#
# A structured tool allows us to define multiple inputs
# with a Pydantic schema.
# ============================================================


# ------------------------------------------------------------
# Define the input schema
# ------------------------------------------------------------

class TitleInput(BaseModel):
    
    topic: str
    
    tone: str = "concise"


# ------------------------------------------------------------
# Define the tool function
# ------------------------------------------------------------

def title_tool_fn(
    topic: str,
    tone: str = "concise"
) -> str:
    
    return f"{tone.title()} Title: {topic} in Practice"


# ------------------------------------------------------------
# Create StructuredTool
# ------------------------------------------------------------

title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="""
    Generate a title given a topic and an optional tone.
    """,
    args_schema=TitleInput
)


# ------------------------------------------------------------
# Create an agent with the structured tool
# ------------------------------------------------------------

structured_agent = create_react_agent(
    model=llm,
    tools=[title_tool]
)


# ------------------------------------------------------------
# Run the structured tool agent
# ------------------------------------------------------------

result = structured_agent.invoke({
    "messages": [
        (
            "user",
            "Make a friendly title about LangGraph tutorials."
        )
    ]
})


print("✅ Result:")
print(result["messages"][-1].content)

C:\Users\khavy\AppData\Local\Temp\ipykernel_3740\193435300.py:51: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  structured_agent = create_react_agent(


GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19h45m4.674231706s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie